# Weekly Aggregation - Cattle Slaughter Week Alignment

This notebook aggregates daily MERRA-2 climate statistics to weekly values aligned with USDA cattle slaughter reporting weeks.

**Purpose:** Create weekly climate data that can be directly correlated with weekly cattle slaughter data.

## Cattle Data Week Structure

- USDA reports cattle slaughter data weekly, with weeks ending on **Saturday**
- Week dates in `cattle_data_clean.csv` represent the **week ending date** (Saturday)
- Example: `1984-01-07` (Saturday) represents the week 1984-01-01 through 1984-01-07

## Aggregation Methods

### Temperature Threshold Metrics (SUM)
Daily data contains **hour counts** (e.g., hours_above_24 = number of hours per day)  
Weekly aggregation: **SUM** across 7 days to get total hours per week

- `hours_above_24`: Daily values 0-10 → Weekly values 0-70 (sum of 7 days × 10 nighttime hours)
- `hours_above_30`: Daily values 0-24 → Weekly values 0-168 (sum of 7 days × 24 hours)

### VPD Metrics (AVERAGE)
Daily data contains **continuous values** (e.g., vpd_mean in kPa)  
Weekly aggregation: **AVERAGE** across 7 days

- `vpd_mean`: Daily average → Weekly average of daily averages

## Output

- **Directory:** `processed_weekly/`
- **Files:** 
  - `weekly_nighttime_recovery.nc` - All weeks, all nighttime metrics
  - `weekly_daytime_heat.nc` - All weeks, all daytime metrics  
  - `weekly_vpd.nc` - All weeks, all VPD metrics
- **Dimensions:** (week, lat, lon)
- **Time coordinate:** Week ending date (Saturday) to match cattle data
- **Period:** 1984-01-07 to 2025-12-27 (filtered to match available climate data through 2025-12-31)

In [1]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Imports complete!")

Imports complete!


In [ ]:
# Configuration - Import from centralized config.py
import sys
from pathlib import Path

# Add project root to path to import config
sys.path.append(str(Path('../..')))  # Up two levels from notebooks/02_data_processing/ to research/

# Import paths from config
from config import (
    CATTLE_DATA_FILE,
    PROCESSED_NIGHTTIME_DIR,
    PROCESSED_DAYTIME_DIR,
    PROCESSED_VPD_DIR,
    PROCESSED_WEEKLY_DIR
)

# Ensure output directory exists
PROCESSED_WEEKLY_DIR.mkdir(exist_ok=True)

print("Project paths loaded from config.py:")
print(f"  Cattle data: {CATTLE_DATA_FILE}")
print(f"  Nighttime data: {PROCESSED_NIGHTTIME_DIR}")
print(f"  Daytime data: {PROCESSED_DAYTIME_DIR}")
print(f"  VPD data: {PROCESSED_VPD_DIR}")
print(f"  Output directory: {PROCESSED_WEEKLY_DIR}")

## Load Cattle Data Week Definitions

Load the cattle slaughter data to extract week ending dates (Saturdays).

In [ ]:
# Load cattle data
df_cattle = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])

# Filter to only include weeks where we have climate data
# Climate data available through 2025-12-31 → last Saturday is 2025-12-27
CLIMATE_DATA_END = pd.Timestamp('2025-12-27')  # Last Saturday with climate data
df_cattle = df_cattle[df_cattle['date'] <= CLIMATE_DATA_END].copy()

# Extract week ending dates
week_ending_dates = df_cattle['date'].values
week_numbers = df_cattle['week'].values

print(f"Total weeks in cattle data (filtered to available climate data): {len(week_ending_dates)}")
print(f"Date range: {week_ending_dates[0]} to {week_ending_dates[-1]}")
print(f"Climate data cutoff: {CLIMATE_DATA_END.date()}\n")

print(f"First 5 weeks:")
for i in range(5):
    week_end = pd.Timestamp(week_ending_dates[i])
    week_start = week_end - pd.Timedelta(days=6)
    week_num = int(week_numbers[i]) if not np.isnan(week_numbers[i]) else 0
    print(f"  Week {week_num:2d}: {week_start.date()} to {week_end.date()} (ending {week_end.day_name()})")

print(f"\nLast 5 weeks:")
for i in range(-5, 0):
    week_end = pd.Timestamp(week_ending_dates[i])
    week_start = week_end - pd.Timedelta(days=6)
    week_num = int(week_numbers[i]) if not np.isnan(week_numbers[i]) else 0
    print(f"  Week {week_num:2d}: {week_start.date()} to {week_end.date()} (ending {week_end.day_name()})")

## Helper Functions

In [4]:
def load_daily_data_for_period(data_dir, start_date, end_date, file_pattern):
    """
    Load daily data files for a date range.
    
    Parameters
    ----------
    data_dir : Path
        Directory containing monthly NetCDF files
    start_date : pd.Timestamp
        Start date (inclusive)
    end_date : pd.Timestamp
        End date (inclusive)
    file_pattern : str
        File pattern with YYYYMM placeholder (e.g., 'nighttime_recovery_{}.nc')
    
    Returns
    -------
    xr.Dataset or None
        Combined dataset for the period, or None if no data available
    """
    # Generate list of months to load
    months = pd.date_range(start_date, end_date, freq='MS')
    
    datasets = []
    for month in months:
        file_path = data_dir / file_pattern.format(f"{month.year}{month.month:02d}")
        if file_path.exists():
            try:
                ds = xr.open_dataset(file_path)
                # Select only dates in our range
                ds = ds.sel(time=slice(start_date, end_date))
                if len(ds.time) > 0:
                    datasets.append(ds)
            except Exception as e:
                print(f"Warning: Error loading {file_path.name}: {e}")
    
    if not datasets:
        return None
    
    # Concatenate all datasets
    return xr.concat(datasets, dim='time')


def aggregate_week_sum(daily_data, week_start, week_end):
    """
    Aggregate daily hour counts to weekly totals (SUM).
    
    Parameters
    ----------
    daily_data : xr.Dataset
        Daily data with time dimension
    week_start : pd.Timestamp
        Week start date (inclusive)
    week_end : pd.Timestamp
        Week end date (inclusive)
    
    Returns
    -------
    dict
        Dictionary of {variable: weekly_sum_array}
    """
    # Select week data
    week_data = daily_data.sel(time=slice(week_start, week_end))
    
    if len(week_data.time) == 0:
        return None
    
    # Sum across time dimension for each variable
    weekly_sums = {}
    for var in week_data.data_vars:
        data = week_data[var]
        
        # Convert timedelta to hours if stored as timedelta
        if data.dtype.kind == 'm':  # 'm' = timedelta
            data = data / np.timedelta64(1, 'h')
        
        # Replace fill values (-1) with 0 before summing
        data = data.where(data != -1, 0)
        
        # Sum and convert to float32
        weekly_sums[var] = data.sum(dim='time').values.astype(np.float32)
    
    return weekly_sums


def aggregate_week_mean(daily_data, week_start, week_end):
    """
    Aggregate daily values to weekly averages (MEAN).
    
    Parameters
    ----------
    daily_data : xr.Dataset
        Daily data with time dimension
    week_start : pd.Timestamp
        Week start date (inclusive)
    week_end : pd.Timestamp
        Week end date (inclusive)
    
    Returns
    -------
    dict
        Dictionary of {variable: weekly_mean_array}
    """
    # Select week data
    week_data = daily_data.sel(time=slice(week_start, week_end))
    
    if len(week_data.time) == 0:
        return None
    
    # Average across time dimension for each variable
    weekly_means = {}
    for var in week_data.data_vars:
        data = week_data[var]
        
        # Convert timedelta to hours if stored as timedelta
        if data.dtype.kind == 'm':  # 'm' = timedelta
            data = data / np.timedelta64(1, 'h')
        
        # Replace fill values (-1) with NaN before averaging
        data = data.where(data != -1)
        
        # Average and convert to float32
        weekly_means[var] = data.mean(dim='time').values.astype(np.float32)
    
    return weekly_means


print("Helper functions defined!")

Helper functions defined!


## Aggregate Nighttime Recovery (SUM)

Sum daily nighttime hour counts to weekly totals.

In [ ]:
output_file = PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc'

if output_file.exists():
    print(f"Output file {output_file.name} already exists.")
    print("Delete it first if you want to regenerate.")
else:
    print("Loading all daily nighttime recovery data...")
    
    # Load all daily data
    start_date = pd.Timestamp(week_ending_dates[0]) - pd.Timedelta(days=6)
    end_date = pd.Timestamp(week_ending_dates[-1])
    
    daily_data = load_daily_data_for_period(
        PROCESSED_NIGHTTIME_DIR, 
        start_date, 
        end_date,
        'nighttime_recovery_{}.nc'
    )
    
    if daily_data is None:
        print("ERROR: No daily nighttime data found!")
    else:
        print(f"Loaded daily data: {len(daily_data.time)} days")
        print(f"Variables: {list(daily_data.data_vars)}")
        
        # Get dimensions
        n_weeks = len(week_ending_dates)
        lat = daily_data.lat.values
        lon = daily_data.lon.values
        n_lat = len(lat)
        n_lon = len(lon)
        
        # Initialize weekly data arrays
        variables = list(daily_data.data_vars)
        weekly_data = {var: np.full((n_weeks, n_lat, n_lon), np.nan, dtype=np.float32) 
                      for var in variables}
        
        # Aggregate each week
        print(f"\nAggregating {n_weeks} weeks (SUM method)...")
        for i, week_end in enumerate(tqdm(week_ending_dates, desc="Weeks")):
            week_end = pd.Timestamp(week_end)
            week_start = week_end - pd.Timedelta(days=6)
            
            week_sums = aggregate_week_sum(daily_data, week_start, week_end)
            
            if week_sums is not None:
                for var in variables:
                    weekly_data[var][i, :, :] = week_sums[var]
        
        # Close daily data to free memory
        daily_data.close()
        
        # Create xarray Dataset
        ds_weekly = xr.Dataset(
            data_vars={var: (['week', 'lat', 'lon'], weekly_data[var]) 
                      for var in variables},
            coords={
                'week': week_ending_dates,
                'lat': lat,
                'lon': lon,
            }
        )
        
        # Add metadata
        ds_weekly.attrs['title'] = 'Weekly Nighttime Recovery Statistics'
        ds_weekly.attrs['description'] = 'Weekly sum of daily nighttime hour counts, aligned with USDA cattle slaughter weeks'
        ds_weekly.attrs['aggregation_method'] = 'SUM (total hours per week)'
        ds_weekly.attrs['week_definition'] = 'Week ending on Saturday (matches cattle_data_clean.csv)'
        ds_weekly.attrs['source'] = 'MERRA-2 nighttime recovery daily statistics'
        ds_weekly.attrs['created'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # Add variable metadata
        for var in variables:
            ds_weekly[var].attrs['long_name'] = f'Weekly total {var}'
            ds_weekly[var].attrs['units'] = 'hours'
            ds_weekly[var].attrs['aggregation'] = 'sum of 7 daily values'
        
        # Add coordinate metadata
        ds_weekly['week'].attrs['long_name'] = 'Week ending date'
        ds_weekly['week'].attrs['description'] = 'Saturday date marking end of 7-day week'
        
        # Save to NetCDF
        encoding = {var: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} 
                   for var in variables}
        ds_weekly.to_netcdf(output_file, encoding=encoding)
        ds_weekly.close()
        
        print(f"\n✓ Saved: {output_file.name}")
        print(f"  Dimensions: {n_weeks} weeks × {n_lat} lat × {n_lon} lon")
        print(f"  Variables: {', '.join(variables)}")

## Aggregate Daytime Heat (SUM)

Sum daily daytime hour counts to weekly totals.

In [6]:
output_file = OUTPUT_DIR / 'weekly_daytime_heat.nc'

if output_file.exists():
    print(f"Output file {output_file.name} already exists.")
    print("Delete it first if you want to regenerate.")
else:
    print("Loading all daily daytime heat data...")
    
    # Load all daily data
    start_date = pd.Timestamp(week_ending_dates[0]) - pd.Timedelta(days=6)
    end_date = pd.Timestamp(week_ending_dates[-1])
    
    daily_data = load_daily_data_for_period(
        DAYTIME_DIR, 
        start_date, 
        end_date,
        'daytime_heat_{}.nc'
    )
    
    if daily_data is None:
        print("ERROR: No daily daytime data found!")
    else:
        print(f"Loaded daily data: {len(daily_data.time)} days")
        print(f"Variables: {list(daily_data.data_vars)}")
        
        # Get dimensions
        n_weeks = len(week_ending_dates)
        lat = daily_data.lat.values
        lon = daily_data.lon.values
        n_lat = len(lat)
        n_lon = len(lon)
        
        # Initialize weekly data arrays
        variables = list(daily_data.data_vars)
        weekly_data = {var: np.full((n_weeks, n_lat, n_lon), np.nan, dtype=np.float32) 
                      for var in variables}
        
        # Aggregate each week
        print(f"\nAggregating {n_weeks} weeks (SUM method)...")
        for i, week_end in enumerate(tqdm(week_ending_dates, desc="Weeks")):
            week_end = pd.Timestamp(week_end)
            week_start = week_end - pd.Timedelta(days=6)
            
            week_sums = aggregate_week_sum(daily_data, week_start, week_end)
            
            if week_sums is not None:
                for var in variables:
                    weekly_data[var][i, :, :] = week_sums[var]
        
        # Close daily data to free memory
        daily_data.close()
        
        # Create xarray Dataset
        ds_weekly = xr.Dataset(
            data_vars={var: (['week', 'lat', 'lon'], weekly_data[var]) 
                      for var in variables},
            coords={
                'week': week_ending_dates,
                'lat': lat,
                'lon': lon,
            }
        )
        
        # Add metadata
        ds_weekly.attrs['title'] = 'Weekly Daytime Heat Statistics'
        ds_weekly.attrs['description'] = 'Weekly sum of daily daytime hour counts, aligned with USDA cattle slaughter weeks'
        ds_weekly.attrs['aggregation_method'] = 'SUM (total hours per week)'
        ds_weekly.attrs['week_definition'] = 'Week ending on Saturday (matches cattle_data_clean.csv)'
        ds_weekly.attrs['source'] = 'MERRA-2 daytime heat daily statistics'
        ds_weekly.attrs['created'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # Add variable metadata
        for var in variables:
            ds_weekly[var].attrs['long_name'] = f'Weekly total {var}'
            ds_weekly[var].attrs['units'] = 'hours'
            ds_weekly[var].attrs['aggregation'] = 'sum of 7 daily values'
        
        # Add coordinate metadata
        ds_weekly['week'].attrs['long_name'] = 'Week ending date'
        ds_weekly['week'].attrs['description'] = 'Saturday date marking end of 7-day week'
        
        # Save to NetCDF
        encoding = {var: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} 
                   for var in variables}
        ds_weekly.to_netcdf(output_file, encoding=encoding)
        ds_weekly.close()
        
        print(f"\n✓ Saved: {output_file.name}")
        print(f"  Dimensions: {n_weeks} weeks × {n_lat} lat × {n_lon} lon")
        print(f"  Variables: {', '.join(variables)}")

Loading all daily daytime heat data...
Loaded daily data: 15337 days
Variables: ['hours_above_25', 'hours_above_30', 'hours_above_35', 'hours_above_40', 'hours_below_neg5', 'hours_below_0', 'hours_below_5']

Aggregating 2191 weeks (SUM method)...


Weeks:   0%|          | 0/2191 [00:00<?, ?it/s]


✓ Saved: weekly_daytime_heat.nc
  Dimensions: 2191 weeks × 51 lat × 95 lon
  Variables: hours_above_25, hours_above_30, hours_above_35, hours_above_40, hours_below_neg5, hours_below_0, hours_below_5


## Aggregate VPD (AVERAGE)

Average daily VPD values to weekly means.

In [7]:
output_file = OUTPUT_DIR / 'weekly_vpd.nc'

if output_file.exists():
    print(f"Output file {output_file.name} already exists.")
    print("Delete it first if you want to regenerate.")
else:
    print("Loading all daily VPD data...")
    
    # Load all daily data
    start_date = pd.Timestamp(week_ending_dates[0]) - pd.Timedelta(days=6)
    end_date = pd.Timestamp(week_ending_dates[-1])
    
    daily_data = load_daily_data_for_period(
        VPD_DIR, 
        start_date, 
        end_date,
        'vpd_{}.nc'
    )
    
    if daily_data is None:
        print("ERROR: No daily VPD data found!")
    else:
        print(f"Loaded daily data: {len(daily_data.time)} days")
        print(f"Variables: {list(daily_data.data_vars)}")
        
        # Get dimensions
        n_weeks = len(week_ending_dates)
        lat = daily_data.lat.values
        lon = daily_data.lon.values
        n_lat = len(lat)
        n_lon = len(lon)
        
        # Initialize weekly data arrays
        variables = list(daily_data.data_vars)
        weekly_data = {var: np.full((n_weeks, n_lat, n_lon), np.nan, dtype=np.float32) 
                      for var in variables}
        
        # Aggregate each week
        print(f"\nAggregating {n_weeks} weeks (MEAN method)...")
        for i, week_end in enumerate(tqdm(week_ending_dates, desc="Weeks")):
            week_end = pd.Timestamp(week_end)
            week_start = week_end - pd.Timedelta(days=6)
            
            week_means = aggregate_week_mean(daily_data, week_start, week_end)
            
            if week_means is not None:
                for var in variables:
                    weekly_data[var][i, :, :] = week_means[var]
        
        # Close daily data to free memory
        daily_data.close()
        
        # Create xarray Dataset
        ds_weekly = xr.Dataset(
            data_vars={var: (['week', 'lat', 'lon'], weekly_data[var]) 
                      for var in variables},
            coords={
                'week': week_ending_dates,
                'lat': lat,
                'lon': lon,
            }
        )
        
        # Add metadata
        ds_weekly.attrs['title'] = 'Weekly VPD Statistics'
        ds_weekly.attrs['description'] = 'Weekly average of daily VPD values, aligned with USDA cattle slaughter weeks'
        ds_weekly.attrs['aggregation_method'] = 'MEAN (average of daily values)'
        ds_weekly.attrs['week_definition'] = 'Week ending on Saturday (matches cattle_data_clean.csv)'
        ds_weekly.attrs['source'] = 'MERRA-2 VPD daily statistics'
        ds_weekly.attrs['created'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # Add variable metadata
        for var in variables:
            ds_weekly[var].attrs['long_name'] = f'Weekly average {var}'
            ds_weekly[var].attrs['units'] = 'kPa'
            ds_weekly[var].attrs['aggregation'] = 'mean of 7 daily values'
        
        # Add coordinate metadata
        ds_weekly['week'].attrs['long_name'] = 'Week ending date'
        ds_weekly['week'].attrs['description'] = 'Saturday date marking end of 7-day week'
        
        # Save to NetCDF
        encoding = {var: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} 
                   for var in variables}
        ds_weekly.to_netcdf(output_file, encoding=encoding)
        ds_weekly.close()
        
        print(f"\n✓ Saved: {output_file.name}")
        print(f"  Dimensions: {n_weeks} weeks × {n_lat} lat × {n_lon} lon")
        print(f"  Variables: {', '.join(variables)}")

Loading all daily VPD data...
Loaded daily data: 15337 days
Variables: ['vpd_mean', 'vpd_min', 'vpd_max']

Aggregating 2191 weeks (MEAN method)...


Weeks:   0%|          | 0/2191 [00:00<?, ?it/s]


✓ Saved: weekly_vpd.nc
  Dimensions: 2191 weeks × 51 lat × 95 lon
  Variables: vpd_mean, vpd_min, vpd_max


## Verify Weekly Data

Quick verification of aggregated weekly data.

In [8]:
# List output files
output_files = sorted(OUTPUT_DIR.glob('*.nc'))
print(f"Weekly output files: {len(output_files)}\n")

for file_path in output_files:
    print(f"\n{'='*60}")
    print(f"File: {file_path.name}")
    print(f"{'='*60}")
    
    ds = xr.open_dataset(file_path)
    print(ds)
    
    # Show sample values for first variable
    first_var = list(ds.data_vars)[0]
    print(f"\nSample values for {first_var}:")
    
    # Get values and handle potential NaN/non-numeric gracefully
    first_week_data = ds[first_var].isel(week=0)
    last_week_data = ds[first_var].isel(week=-1)
    
    # Compute statistics
    try:
        fw_min = float(first_week_data.min().values)
        fw_max = float(first_week_data.max().values)
        fw_mean = float(first_week_data.mean().values)
        print(f"  First week: min={fw_min:.2f}, max={fw_max:.2f}, mean={fw_mean:.2f}")
    except (ValueError, TypeError) as e:
        print(f"  First week: Could not compute statistics ({e})")
    
    try:
        lw_min = float(last_week_data.min().values)
        lw_max = float(last_week_data.max().values)
        lw_mean = float(last_week_data.mean().values)
        print(f"  Last week:  min={lw_min:.2f}, max={lw_max:.2f}, mean={lw_mean:.2f}")
    except (ValueError, TypeError) as e:
        print(f"  Last week: Could not compute statistics ({e})")
    
    # Check data type
    print(f"  Data type: {ds[first_var].dtype}")
    print(f"  Shape: {ds[first_var].shape}")
    
    ds.close()

print(f"\n{'='*60}")
print("✓ Weekly aggregation complete!")
print(f"{'='*60}")

Weekly output files: 3


File: weekly_daytime_heat.nc
<xarray.Dataset> Size: 594MB
Dimensions:           (week: 2191, lat: 51, lon: 95)
Coordinates:
  * week              (week) datetime64[ns] 18kB 1984-01-07 ... 2025-12-27
  * lat               (lat) float64 408B 24.0 24.5 25.0 25.5 ... 48.0 48.5 49.0
  * lon               (lon) float64 760B -125.0 -124.4 -123.8 ... -66.88 -66.25
Data variables:
    hours_above_25    (week, lat, lon) timedelta64[ns] 85MB ...
    hours_above_30    (week, lat, lon) timedelta64[ns] 85MB ...
    hours_above_35    (week, lat, lon) timedelta64[ns] 85MB ...
    hours_above_40    (week, lat, lon) timedelta64[ns] 85MB ...
    hours_below_neg5  (week, lat, lon) timedelta64[ns] 85MB ...
    hours_below_0     (week, lat, lon) timedelta64[ns] 85MB ...
    hours_below_5     (week, lat, lon) timedelta64[ns] 85MB ...
Attributes:
    title:               Weekly Daytime Heat Statistics
    description:         Weekly sum of daily daytime hour counts, aligned wit...
   

## Summary

This notebook successfully:

1. ✓ Loaded USDA cattle slaughter week definitions (Saturday ending dates)
2. ✓ Aggregated daily nighttime recovery metrics to weekly totals (SUM)
3. ✓ Aggregated daily daytime heat metrics to weekly totals (SUM)
4. ✓ Aggregated daily VPD metrics to weekly averages (MEAN)

**Output files:**
- `processed_weekly/weekly_nighttime_recovery.nc`
- `processed_weekly/weekly_daytime_heat.nc`
- `processed_weekly/weekly_vpd.nc`

**Next steps:**
1. Update analysis notebooks to load these weekly files
2. Merge with cattle_data_clean.csv on week ending date
3. Perform correlation analysis between climate and cattle slaughter